# Search engine with tools and agnts

In [1]:
from langchain_community.tools import ArxivQueryRun,WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper,ArxivAPIWrapper



C:\Users\DELL ADMIN\AppData\Local\Temp\ipykernel_18080\4287049590.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools import ArxivQueryRun,WikipediaQueryRun


In [2]:
api_wrapper_wiki = WikipediaAPIWrapper(top_k_results=1,doc_content_chars_max=250)
wiki = WikipediaQueryRun(api_wrapper=api_wrapper_wiki)
wiki.name

'wikipedia'

In [3]:
api_wrapper_arxiv = ArxivAPIWrapper(top_k_results=1,doc_content_chars_max=250)
arxiv = ArxivQueryRun(api_wrapper=api_wrapper_arxiv)
arxiv.name

'arxiv'

In [4]:
tools = [wiki,arxiv]

In [5]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS


c:\langchain_projects\simple_chatbot\cbot\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
USER_AGENT environment variable not set, consider setting it to identify your requests.


In [6]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["HF_TOKEN"] = os.getenv("HF_TOKEN")

In [7]:
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6567.95it/s]


In [8]:
loader = WebBaseLoader("https://python.langchain.com/docs/introduction/")
docs = loader.load()
documents = RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200).split_documents(docs)
vectordb = FAISS.from_documents(documents,embeddings)
retriever = vectordb.as_retriever()
retriever

VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x00000194B908FEF0>, search_kwargs={})

In [9]:
from langchain_core.tools.retriever import create_retriever_tool
retriever_tool = create_retriever_tool(retriever,"langsmith-serach","serach any information about langsmith")
retriever_tool 

StructuredTool(name='langsmith-serach', description='serach any information about langsmith', args_schema=<class 'langchain_core.tools.retriever.RetrieverInput'>, func=<function create_retriever_tool.<locals>.func at 0x0000019486073880>, coroutine=<function create_retriever_tool.<locals>.afunc at 0x00000194B8EA2AC0>)

In [10]:
tools = [wiki,arxiv,retriever_tool]

In [11]:
tools

[WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper(wiki_client=<module 'wikipedia' from 'c:\\langchain_projects\\simple_chatbot\\cbot\\Lib\\site-packages\\wikipedia\\__init__.py'>, top_k_results=1, lang='en', load_all_available_meta=False, doc_content_chars_max=250)),
 ArxivQueryRun(api_wrapper=ArxivAPIWrapper(arxiv_search=<class 'arxiv.Search'>, arxiv_exceptions=(<class 'arxiv.ArxivError'>, <class 'arxiv.UnexpectedEmptyPageError'>, <class 'arxiv.HTTPError'>), top_k_results=1, ARXIV_MAX_QUERY_LENGTH=300, continue_on_failure=False, load_max_docs=100, load_all_available_meta=False, doc_content_chars_max=250)),
 StructuredTool(name='langsmith-serach', description='serach any information about langsmith', args_schema=<class 'langchain_core.tools.retriever.RetrieverInput'>, func=<function create_retriever_tool.<locals>.func at 0x0000019486073880>, coroutine=<function create_retriever_tool.<locals>.afunc at 0x00000194B8EA2AC0>)]

In [12]:
## run all this tools with agents and llm
from langchain_groq import ChatGroq

grog_api_key = os.getenv("GROQ_API_KEY")

llm = ChatGroq(groq_api_key=grog_api_key,model_name="llama-3.1-8b-instant")

In [26]:
from langsmith import Client

client = Client()
prompt = client.pull_prompt("hwchase17/openai-tools-agent",dangerously_pull_public_prompt=True)
prompt.messages

[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You are a helpful assistant'), additional_kwargs={}),
 MessagesPlaceholder(variable_name='chat_history', optional=True),
 HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={}),
 MessagesPlaceholder(variable_name='agent_scratchpad')]

In [29]:
from langchain_classic.agents import create_openai_tools_agent, AgentExecutor

# Create the agent
agent = create_openai_tools_agent(
    llm=llm,
    tools=tools,
    prompt=prompt
)

# Create the executor
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True
)

# Invoke
response = agent_executor.invoke(
    {"input": "capital city of Telangana"}
)

print(response["output"])



> Entering new AgentExecutor chain...

Invoking: `wikipedia` with `{'query': 'Capital of Telangana'}`


Page: Hyderabad
Summary: Hyderabad is the capital and largest city of the Indian state of Telangana. It occupies 650 km2 (250 sq mi) on the Deccan Plateau along the banks of the Musi River, in the northern part of South India. With an average altitud
Invoking: `wikipedia` with `{'query': 'Hyderabad, India population'}`


Page: Hyderabad
Summary: Hyderabad is the capital and largest city of the Indian state of Telangana. It occupies 650 km2 (250 sq mi) on the Deccan Plateau along the banks of the Musi River, in the northern part of South India. With an average altitudThe capital city of Telangana is Hyderabad. As of 2020, the population of Hyderabad is approximately 10 million.

> Finished chain.
The capital city of Telangana is Hyderabad. As of 2020, the population of Hyderabad is approximately 10 million.
